# Ⅰ第2回 演習2 学習済み3モデルの4指標

3 つの学習済みモデル（MLP：埋め込み平均＋全結合，CNN：1次元畳み込み，Transformer：小型の Transformer）を動かし，
**正解率・損失・推論時間・メモリ** の 4 指標を同じ関数で測る．
3 モデルはパラメータ数を約 30 万に揃えてあり，同じデータ（SST-2 映画レビューの肯定/否定）で同じ回数だけ学習してある．

**実行する前に，下の「予測」セルを埋める．** 実行時間の目安は 5 分以内．

## (0) グループと役割の設定

In [1]:
# ===== (0) グループと役割の設定 =====
GROUP_ID = 12                  # ← 自分のグループ番号 (1〜27) に書き換える
MEMBER_ROLE = "verifier"   # implementer / verifier / recorder / presenter のいずれか（3 人・5 人グループの verifier は "verifier"）

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

作業フォルダ: /Users/hinata/AI_TD


## GW：3つのモデルの構成を読み解く（グループ・提出）

授業ページ演習2の図と語群を見て，グループで決めた**記号**を ＿＿ に書く（このセルをダブルクリックして書き込む．各記号は1回だけ使う）．

| 仕組み | しくみ | 得意 | 不得意 |
|---|---|---|---|
| 全結合（MLP） | ＿＿ | ＿＿ | ＿＿ |
| 畳み込み（CNN） | ＿＿ | ＿＿ | ＿＿ |
| 自己注意（Transformer） | ＿＿ | ＿＿ | ＿＿ |

語群
- しくみ：ア となり合う数語（例：3語）を1つの「窓」として，窓の中の語に重みを掛けて足す。窓を1語ずつずらしながら，文のはじめから終わりまで同じ重みで計算する ／ イ 各単語が，ほかの全単語との関係を計算して情報を混ぜる ／ ウ 入力のすべての値にそれぞれ重みを掛けて足し，出力の値を1つ作る（出力の数だけくり返す）。入力全体の情報をまとめて，新しい特徴や，最後の「肯定か否定か」の点数を作るため
- 得意：エ 離れた語どうしの関係も見られる ／ オ 計算が単純で速い ／ カ 近くの並びのパターンを見つける
- 不得意：キ 窓より遠い語どうしの関係が見えにくい ／ ク 語順・位置を区別できない ／ ケ 文が長いと，計算とメモリが大きく増える

## 予測（各自・実行する前に書く）

このセルをダブルクリックして，「予測」の列に MLP・CNN・Transformer のどれかと理由（一言）を書く．
「実際」の列は，(10) の結果を見てから書く．

| 問い | 予測 | 理由（一言） | 実際 |
|---|---|---|---|
| 正解率が最も高いのは | ＿＿ | ＿＿ | ＿＿ |
| 損失が最も低いのは | ＿＿ | ＿＿ | ＿＿ |
| 1000文あたり最も速いのは | ＿＿ | ＿＿ | ＿＿ |
| 1文あたり最も速いのは | ＿＿ | ＿＿ | ＿＿ |
| 推論中のメモリが最も少ないのは | ＿＿ | ＿＿ | ＿＿ |

## (1) コードテンプレート①：ライブラリのインポート

In [ ]:
import time
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from common import ASSETS, load_config, memory
from common.device import get_device, synchronize
from common.data import load_corpus, decode
from common.models import build_model, count_params
from common.logger import ResultLogger
from common.todo import todo_check, checkpoint

## (2) コードテンプレート⑤：使用デバイスの選択

演習1 と同じ．`get_device()` はテンプレート⑤にメモリ上限を加えたもの．

In [ ]:
device = get_device()
print(f"使用デバイス: {device}")

## (3) コードテンプレート②③-テキスト：データ変換とデータセットの作成

教科書のテンプレート②（データ変換）と③（乱数固定とデータセット作成）に対応する．
画像では `transforms` が行う変換を，テキストでは「トークン化 → ID 化 → 固定長化」が担う（`common/data.py`．授業冒頭の `setup/download_assets_day2.py` で取得済み）．
評価には **テストデータ**（1,821 文．訓練にも検証にも使っていない文）を使う．

In [ ]:
# コードテンプレート③：乱数固定（教科書と同じ 3 行）
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

cfg = load_config()
corpus = load_corpus(cfg)
X_test = torch.from_numpy(corpus["X_test"])     # (1821, 40) トークン ID．0 はパディング
y_test = torch.from_numpy(corpus["y_test"])     # 0 = 否定, 1 = 肯定
print(f"訓練データ数: {len(corpus['X_train'])} 文")
print(f"検証データ数: {len(corpus['X_dev'])} 文")
print(f"テストデータ数: {len(X_test)} 文")
print("語彙:", corpus["vocab_size"], "| 系列長:", corpus["max_len"])
for i in range(3):
    print(y_test[i].item(), "|", decode(X_test[i], corpus["vocab"]))

## (3-2) データを眺める

測る前に，テストデータがどんな文かを見る．`<unk>` は語彙（よく出る 5,000 語）に入らなかった語．
見ながら考える：肯定と否定を分ける手がかりになりそうな語は何か．語順が大事になりそうな文はあるか．

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]   # Mac の日本語フォント（文字化け防止）

# ラベルの割合（0 = 否定，1 = 肯定）
print("肯定の割合:", f"{y_test.float().mean().item():.1%}", f"（{int(y_test.sum())} / {len(y_test)} 文）")

# 肯定・否定の例を 5 文ずつ（実行するたびに変わる）
g = torch.Generator().manual_seed(int(time.time()))
for label, name in [(1, "肯定"), (0, "否定")]:
    idx = torch.nonzero(y_test == label).flatten()
    print(f"\n--- {name} の例 ---")
    for i in idx[torch.randperm(len(idx), generator=g)[:5]]:
        print(" ", decode(X_test[i], corpus["vocab"]))

# 文の長さ（語数）と <unk> の割合
lengths = (X_test != 0).sum(dim=1)
print(f"\n平均 {lengths.float().mean():.1f} 語，最長 {lengths.max().item()} 語（40 語で切っている）")
print(f"<unk> の割合: {((X_test == 1).sum() / (X_test != 0).sum()).item():.1%}")
plt.hist(lengths.numpy(), bins=range(0, 42, 2))
plt.xlabel("文の長さ（語数）"); plt.ylabel("文の数"); plt.title("テストデータの文の長さ")
plt.show()

## (4) コードテンプレート④⑥-計測用：ミニバッチと形状の確認

教科書のテンプレート④は `DataLoader` でミニバッチを作る．ここでは時間を測るときに **データ転送の時間を混ぜない** ため，先にテンソルをデバイスへ載せてから分割する（`DataLoader` を使うと毎回 CPU→GPU の転送が入り，モデルの速さではなく転送の速さを測ってしまう）．
テンプレート⑥にならい，ミニバッチの形状を確認する．

In [ ]:
# コードテンプレート④-計測用
BATCH_SIZE = 64
test_batches = [(X_test[i:i + BATCH_SIZE].to(device), y_test[i:i + BATCH_SIZE].to(device))
                for i in range(0, len(X_test), BATCH_SIZE)]

# コードテンプレート⑥：ミニバッチの形状確認
tokens, labels = test_batches[0]
print("ミニバッチサイズ:", len(tokens))
print("系列長:", tokens.shape[1])
print("ミニバッチ数:", len(test_batches))
print(tokens.shape)

## (5) コードテンプレート⑦-学習済みモデル：モデルのロードと評価モード

教科書のテンプレート⑦（モデル定義）の学習済み版．8 章「学習済みモデルのロード」と同じ手順で，重みファイルに入っている構造の情報（隠れ幅など）から同じ構造を組み立て，`load_state_dict` で学習済みパラメータを読み込む．
推論だけを行うので `model.eval()` で **評価モード** にする（Dropout などを無効化する）．

**TODO 1 は 2 か所**：コードの中の `▼ TODO 1-1`（`ckpt = ...` の行）と `▼ TODO 1-2`（`model.load_state_dict(...)` の行）の「...」を書き換える．

このセルで 3 つの学習済みモデル（MLP・CNN・Transformer）を読み込み，パラメータ数がそろっていることを確かめる．(6)〜(8) はそのうち MLP で関数の動作を確かめ，(9) で 3 つとも測る．

In [ ]:
NAMES = {"A1": "MLP", "A2": "CNN", "A3": "Transformer"}   # 重みファイルの記号 → 表示名

def load_pretrained(arch):
    path = ASSETS / "weights" / f"day1_{arch}.pt"      # 重みファイル（.pt）の場所
    # ① 重みファイルを読み込む．ckpt は辞書で，重みの数値（"state_dict"）と形の情報（hidden など）が入っている
    # ▼ TODO 1-1：右の「...」を消して，torch.load で path を読む式を書く（map_location="cpu" を付ける）
    ckpt = ...
    # ② 同じ形のネットワークを用意する（ファイルに保存してある情報から組み立てる．このときの重みはランダム）
    model, info = build_model(arch, ckpt["info"]["target"], ckpt["vocab_size"],
                              num_classes=ckpt["num_classes"], max_len=ckpt["max_len"], d=ckpt["embed_dim"])
    # ③ 用意したネットワークに，学習済みの重みの数値を当てはめる
    # ▼ TODO 1-2：かっこの中の「...」を消して，ckpt の中の重みの数値（ckpt["state_dict"]）を書く
    model.load_state_dict(...)
    # ④ 評価モードにする
    model.eval()
    return model.to(device), info

todo_check("TODO 1")                                  # 「...」が残っていたら止める

print("3 つの学習済みモデル")
for arch in ["A1", "A2", "A3"]:
    m, info = load_pretrained(arch)
    print(" ", NAMES[arch], "params =", f"{count_params(m):,}", "| hidden =", info["hidden"])

## (5-2) 討議（3 分）：hidden とは何か

(5) の出力では，3 つともパラメータ数は約 30 万なのに，`hidden` が 357・152・999 とばらばら．下のセルで各層の重みの形と数を見て，グループで話す．

1. `hidden` は，それぞれのモデルで何の数か（層の幅？ 窓の数？ …）
2. 同じ約 30 万にそろえたのに，なぜ `hidden` の値が 3 つで違うのか（`hidden` を 1 増やすと，重みはいくつ増えるか）
3. 3 つに共通する大きな重みはどれか．それは何のための重みか
4. パラメータ数が同じなら，計算の量やメモリも同じになりそうか（→ (9) で確かめる）

In [ ]:
# 見たいモデルを選んで，層ごとの重みの形と数を表示する
SHOW = "A1"          # "A1"（MLP）/ "A2"（CNN）/ "A3"（Transformer）
m, info = load_pretrained(SHOW)
print(NAMES[SHOW], "| hidden =", info["hidden"], "| 合計", f"{count_params(m):,}")
for name, p in m.named_parameters():
    print(f"  {name:42s} {str(tuple(p.shape)):16s} {p.numel():>9,}")

## ✋ 止まる 1：ここから先へは，グループ全員がそろってから進む

**問い**：hidden は 3 つのモデルでそれぞれ何の数か．同じ約 30 万にそろえたのに，なぜ値が違うのか

1. **presenter**：問いを読み上げ，話し合いを回す
2. **implementer**：答えの根拠を，コードの行と出力を指して説明する（「この行がこう計算したので，この数字になった」）．
   根拠になる所：(5-2) の出力で一番大きい重みの行（MLP の 357×357，CNN の 152×152×5 など）．hidden がどの形に入っているか．最後に，次に書く TODO が何をする行かを予告する
3. **verifier**：全員の出力の数値が同じか，桁がおかしくないかを確かめる
4. **recorder**：根拠を含めて答えを 1〜2 文にまとめ，全員に伝える
5. 全員が下のセルの `CP1 = ""` の `" "` の間に，その答えを日本語で書いて実行する．**10 文字より短い（空のままも含む）とエラーになって止まる**

In [ ]:
# 問い：hidden は 3 つのモデルでそれぞれ何の数か．同じ約 30 万にそろえたのに，なぜ値が違うのか
# ▼ グループで決めた答えを，下の " " の中に日本語で書く（1〜2 文）
#   書き方の例：CP1 = "〇〇だから，△△になると考えた．"
#   注意：CP1 に入れた文字列が 10 文字より短い（空のままも含む）と，エラーになって先へ進めない
CP1 = ""
checkpoint(CP1, "止まる 1")

## (6) コードテンプレート⑧⑫-4指標：損失関数と正解率

(6)〜(8) は 3 つのうち MLP で関数が正しく動くかを確かめる．3 つとも測るのは (9)．

第1回 GW3 のテンプレート⑧（損失関数 `criterion`）と⑫（テストデータでの評価）をそのまま使い，正解率（Accuracy）に加えて損失も集計する．
正解率は「正解した割合」，損失は「正解ラベルに対する確信の低さ」（交差エントロピー）．**同じモデルでも 2 つは違う順位になりうる**．

In [ ]:
criterion = nn.CrossEntropyLoss(reduction="sum")   # 損失関数（sum：ミニバッチの文の損失を合計で返す）

def evaluate(model, batches):
    model.eval()                                      # 評価モード
    correct, total, total_loss = 0, 0, 0.0            # 正解した数・文の数・損失の合計を 0 から数える
    with torch.no_grad():                             # 推論だけなので勾配は計算しない
        for tokens, labels in batches:                # ミニバッチ（64 文）ずつ取り出す
            # ① 順伝播：各文の「否定・肯定」の点数を出す（形は 文の数 × 2）
            outputs = model(tokens)
            # ② 損失：予測と正解のずれを足していく（最後に文の数で割る）
            total_loss += criterion(outputs, labels).item()
            # ③ TODO 2-1：点数が大きいほうを予測クラスにする（torch.max(outputs.data, 1) は「最大値, その位置」の 2 つを返す）
            _, predicted = ...
            # ④ TODO 2-2：このミニバッチの文の数を total に足す（labels.size(0) が文の数）
            total += ...
            # ⑤ TODO 2-3：予測と正解が一致した数を correct に足す（一致した所が True になる．.sum().item() で数にする）
            correct += ...
    if not (total > 0):
        raise RuntimeError("TODO が未記入です：ループ内で total と correct を更新すること")
    return correct / total, total_loss / total        # 正解率と，1 文あたりの損失

todo_check("TODO 2")                                  # 「...」が残っていたら止める

model, _ = load_pretrained("A1")          # 3 つのうち MLP で動作を確かめる
acc, loss = evaluate(model, test_batches)
print(f"MLP: Accuracy: {acc*100:.2f}%  Loss: {loss:.4f}")

## (7) 計測：推論時間

(6)〜(8) は 3 つのうち MLP で関数が正しく動くかを確かめる．3 つとも測るのは (9)．

演習1 で見たとおり **同期** が必要である．また 1 回では変動する（GPU のクロックが上下する）ので，
**最低 2 秒ぶん繰り返して中央値** を取る（演習1 の `timed_loop` と同じ作法）．
「1000 文あたりの ms」（スループット）と「1 文だけ入れたときの ms」（レイテンシ）は **別の指標** である．

In [ ]:
def measure_time(model, batches, min_sec=2.0, min_repeats=5):
    """全テストデータを流す時間 → 1000 文あたりの ms（中央値）"""
    model.eval()
    n = sum(len(t) for t, _ in batches)
    times = []
    with torch.no_grad():
        for tokens, _ in batches:                     # ウォームアップ 1 周
            model(tokens)
        synchronize(device)
        t_start = time.perf_counter()
        while len(times) < min_repeats or time.perf_counter() - t_start < min_sec:
            synchronize(device)                       # ① それまでに頼んだ計算を終わらせる
            t0 = time.perf_counter()                  # ② 時計スタート
            for tokens, _ in batches:                 # ③ 測りたい処理（全テストデータを流す）
                model(tokens)
            # ④ TODO 3: ③の計算が全部終わるまで待つ（同期する）．忘れてもエラーは出ない．値が小さくなるだけ
            times.append(time.perf_counter() - t0)    # ⑤ 時計ストップ
    return float(np.median(times)) / n * 1000 * 1000

def measure_latency(model, batches, n=200):
    """1 文ずつ入れたときの 1 文あたり ms（中央値）"""
    model.eval()
    X = batches[0][0]                                 # 先頭バッチから 1 文ずつ取り出す（デバイス上）
    times = []
    with torch.no_grad():
        for i in range(n):
            x = X[i % len(X):i % len(X) + 1]
            synchronize(device)                       # ① それまでに頼んだ計算を終わらせる
            t0 = time.perf_counter()                  # ② 時計スタート
            model(x)                                  # ③ 測りたい処理（1 文だけ流す）
            # ④ TODO 3: ③の計算が全部終わるまで待つ（同期する）．忘れてもエラーは出ない．値が小さくなるだけ
            times.append(time.perf_counter() - t0)    # ⑤ 時計ストップ
    return float(np.median(times)) * 1000

print(f"MLP: {measure_time(model, test_batches):.2f} ms / 1000文,  {measure_latency(model, test_batches):.3f} ms / 1文")

# 判定：同期を忘れると，MPS では 1 文あたりの時間が 0.05 ms を切るほど小さく出る
if device.type == "mps" and measure_latency(model, test_batches) < 0.05:
    print("NG：TODO 3 の同期が抜けている可能性が高い")

## (8) 計測：メモリ

(6)〜(8) は 3 つのうち MLP で関数が正しく動くかを確かめる．3 つとも測るのは (9)．

「モデルの大きさ」（重みのバイト数）と「推論中のピーク」（途中の計算結果が占める量）は別物．
3 モデルはパラメータ数が同じなので **重みの大きさは同じ**．違いが出るのは推論中のピークである．
`forward_peak_mb` は各層の出力直後に使用量を読み，最大値を返す（裏でサンプリングする方式は数十 ms の処理では取りこぼす）．

In [ ]:
def measure_memory(model, batches):
    model_mb = count_params(model) * 4 / 2**20              # float32 = 4 バイト
    def run():
        with torch.no_grad():
            for tokens, _ in batches:
                model(tokens)
    peak_mb = memory.forward_peak_mb(model, run)             # 推論中に増えた分のピーク
    return model_mb, peak_mb

print("MLP: model %.2f MB, peak %.2f MB" % measure_memory(model, test_batches))

## (9) 4指標：3 モデルの同時測定

同じ関数で 3 モデルを測り，1 枚の表にする．
GPU の最初の遅い実行（演習1）が最初に測るモデルにだけ混ざらないよう，測定前に 2 秒 GPU を回してから始める．

In [ ]:
from common.bench import warm_up
warm_up(device, sec=2.0)                       # 最初の遅い実行を先に済ませる

rows = []
for arch in ["A1", "A2", "A3"]:
    model, info = load_pretrained(arch)
    t0 = time.perf_counter()
    acc, loss = evaluate(model, test_batches)
    ms1000 = measure_time(model, test_batches)
    lat = measure_latency(model, test_batches)
    model_mb, peak_mb = measure_memory(model, test_batches)
    rows.append({"model": NAMES[arch], "params": count_params(model),
                 "accuracy": acc, "loss": loss,
                 "infer_ms_per_1000": ms1000, "latency_ms_b1": lat,
                 "model_mb": model_mb, "peak_mem_mb": peak_mb,
                 "elapsed_sec": time.perf_counter() - t0})
table = pd.DataFrame(rows).set_index("model")    # 計算と CSV は英語の列名のまま使う

# 表示だけ日本語の列名にする
JA = {"params": "パラメータ数", "accuracy": "正解率", "loss": "損失",
      "infer_ms_per_1000": "1000文の時間(ms)", "latency_ms_b1": "1文の時間(ms)",
      "model_mb": "重みの大きさ(MB)", "peak_mem_mb": "推論中のピーク(MB)", "elapsed_sec": "測定にかかった時間(秒)"}
table.rename(columns=JA).rename_axis("モデル").round(4)

## (9-2) バッチサイズを変えて，推論中のメモリを測る（verifier が担当．全員が実行してもよい）

(9) はバッチサイズ 64 で測った．テストデータ 1,821 文を 64 文ずつのミニバッチに分け，29 回に分けて流した（バッチサイズ＝ミニバッチサイズ．授業ページ演習2の図）．一度に流す文の数を増やすと，推論中のメモリはどう変わるか．
1821 はテストデータの全文を一度に流す場合．実行は 1 分ほど．

1. バッチを 8 倍（64 → 512）にすると，ピークは何倍になったか
2. 増えた量（MB）が一番大きいモデルはどれか．GW の「不得意」と結び付けて説明できるか
3. 長い文や大量の文を一度に入れると重くなる，という話は LLM にも当てはまるか

In [ ]:
BATCH_SIZES = [64, 512, 1821]          # 1821 = テストデータの全文を一度に流す
peak_rows = {}
for arch in ["A1", "A2", "A3"]:
    model, _ = load_pretrained(arch)
    peaks = []
    for bs in BATCH_SIZES:
        batches = [(X_test[i:i + bs].to(device), y_test[i:i + bs].to(device))
                   for i in range(0, len(X_test), bs)]
        peaks.append(measure_memory(model, batches)[1])    # (8) の関数．2 つ目の戻り値が推論中のピーク
    peak_rows[NAMES[arch]] = peaks

peak_table = pd.DataFrame(peak_rows, index=[f"バッチ {b}" for b in BATCH_SIZES]).T
print("推論中のピークメモリ（MB）")
display(peak_table.rename_axis("モデル").round(1))
print("バッチ 64 → 512 で何倍：", (peak_table.iloc[:, 1] / peak_table.iloc[:, 0]).round(1).to_dict())

## ✋ 止まる 2：ここから先へは，グループ全員がそろってから進む

**問い**：バッチサイズを大きくすると推論中のメモリはどう変わったか．増えた量が一番大きいモデルはどれで，なぜか

1. **presenter**：問いを読み上げ，話し合いを回す
2. **implementer**：答えの根拠を，コードの行と出力を指して説明する（「この行がこう計算したので，この数字になった」）．
   根拠になる所：(9-2) のコードの `X_test[i:i + bs]`（1 回に流す文の数）と，出力の表の増え方．最後に，次に書く TODO が何をする行かを予告する
3. **verifier**：全員の出力の数値が同じか，桁がおかしくないかを確かめる
4. **recorder**：根拠を含めて答えを 1〜2 文にまとめ，全員に伝える
5. 全員が下のセルの `CP2 = ""` の `" "` の間に，その答えを日本語で書いて実行する．**10 文字より短い（空のままも含む）とエラーになって止まる**

In [ ]:
# 問い：バッチサイズを大きくすると推論中のメモリはどう変わったか．増えた量が一番大きいモデルはどれで，なぜか
# ▼ グループで決めた答えを，下の " " の中に日本語で書く（1〜2 文）
#   書き方の例：CP2 = "〇〇だから，△△になると考えた．"
#   注意：CP2 に入れた文字列が 10 文字より短い（空のままも含む）と，エラーになって先へ進めない
CP2 = ""
checkpoint(CP2, "止まる 2")

## (10) 予測との照合

冒頭の「予測」セルの「実際」の列を埋める．**外れた予測** から討議で話す．

In [ ]:
print("最も正解率が高い:", table["accuracy"].idxmax())
print("最も損失が低い:", table["loss"].idxmin())
print("最も速い(1000文):", table["infer_ms_per_1000"].idxmin())
print("最も速い(1文)  :", table["latency_ms_b1"].idxmin())
print("最も省メモリ    :", table["peak_mem_mb"].idxmin())

## ✋ 止まる 3：ここから先へは，グループ全員がそろってから進む

**問い**：外れた予測はどれか．なぜ外れたと考えるか

1. **presenter**：問いを読み上げ，話し合いを回す
2. **implementer**：答えの根拠を，コードの行と出力を指して説明する（「この行がこう計算したので，この数字になった」）．
   根拠になる所：(10) の出力の各行が，(9) の表のどの列を比べた結果か．最後に，次に書く TODO が何をする行かを予告する
3. **verifier**：全員の出力の数値が同じか，桁がおかしくないかを確かめる
4. **recorder**：根拠を含めて答えを 1〜2 文にまとめ，全員に伝える
5. 全員が下のセルの `CP3 = ""` の `" "` の間に，その答えを日本語で書いて実行する．**10 文字より短い（空のままも含む）とエラーになって止まる**

In [ ]:
# 問い：外れた予測はどれか．なぜ外れたと考えるか
# ▼ グループで決めた答えを，下の " " の中に日本語で書く（1〜2 文）
#   書き方の例：CP3 = "〇〇だから，△△になると考えた．"
#   注意：CP3 に入れた文字列が 10 文字より短い（空のままも含む）と，エラーになって先へ進めない
CP3 = ""
checkpoint(CP3, "止まる 3")

## (11) 記録：CSV に記録

演習3 がこの CSV を読む．

In [ ]:
logger = ResultLogger(GROUP_ID, MEMBER_ROLE, course="c1", day="d2", exercise="ex2", device=device)
for name, r in table.iterrows():                      # name は MLP / CNN / Transformer
    logger.log_many({k: r[k] for k in ["accuracy", "loss", "infer_ms_per_1000", "latency_ms_b1",
                                       "model_mb", "peak_mem_mb", "params"]},
                    condition=name, seed=0, elapsed_sec=r["elapsed_sec"])
table.to_csv(logger.path.with_name(f"table_c1_d2_ex2_{logger.group_id}.csv"))
print("書き込み先:", logger.path)
pd.read_csv(logger.path).tail(21)                     # 3 モデル × 7 項目（実行するたびに行が増える）

## (11-2) グループ全員の結果を並べる

正解率は全員同じになるはず．速さとメモリはマシンで変わる．全員の CSV を集めて並べる．

1. 自分の `results/c1_d2_ex2_<グループ>.csv` を，グループ全員に送る（Slack・AirDrop など）
2. 受け取った CSV は，すべて `results/ex2_shared/` フォルダに入れる（無ければ作る．同じ名前のファイルは末尾に `_2` などを付けて別名にする）
3. 下のセルを実行する（集まるたびに何回実行してもよい）

In [ ]:
import glob
files = glob.glob(str(ROOT / "results" / "c1_d2_ex2_*.csv")) + glob.glob(str(ROOT / "results" / "ex2_shared" / "*.csv"))
everyone = pd.concat([pd.read_csv(f) for f in files]).drop_duplicates()
everyone["group_id"] = everyone["group_id"].astype(str).str.zfill(2)
everyone = everyone[(everyone["group_id"] == f"{GROUP_ID:02d}") & everyone["condition"].isin(["MLP", "CNN", "Transformer"])]
# 1 人（役割 × マシン）につき，いちばん新しい結果を使う
latest = everyone.sort_values("timestamp").groupby(["member_role", "machine", "condition", "metric_name"]).tail(1)
show = latest[latest["metric_name"].isin(["accuracy", "infer_ms_per_1000", "peak_mem_mb"])]
wide = show.pivot_table(index=["member_role", "machine"], columns=["metric_name", "condition"], values="metric_value")
wide = wide.rename(columns={"accuracy": "正解率", "infer_ms_per_1000": "1000文の時間(ms)", "peak_mem_mb": "推論中のピーク(MB)"}, level=0)
wide.index.names = ["役割", "マシン"]
print(f"集まった人数：{len(wide)} 人")
wide.round(4)

## ✋ 止まる 4：ここから先へは，グループ全員がそろってから進む

**問い**：全員の結果を比べて，正解率・速さ・メモリはそれぞれ同じだったか．違ったものは何が原因か

1. **presenter**：問いを読み上げ，話し合いを回す
2. **implementer**：答えの根拠を，コードの行と出力を指して説明する（「この行がこう計算したので，この数字になった」）．
   根拠になる所：(11-2) の表の列．計算が決まっていて同じになる値と，マシン（チップ）で変わる値．最後に，次に書く TODO が何をする行かを予告する
3. **verifier**：全員の出力の数値が同じか，桁がおかしくないかを確かめる
4. **recorder**：根拠を含めて答えを 1〜2 文にまとめ，全員に伝える
5. 全員が下のセルの `CP4 = ""` の `" "` の間に，その答えを日本語で書いて実行する．**10 文字より短い（空のままも含む）とエラーになって止まる**

In [ ]:
# 問い：全員の結果を比べて，正解率・速さ・メモリはそれぞれ同じだったか．違ったものは何が原因か
# ▼ グループで決めた答えを，下の " " の中に日本語で書く（1〜2 文）
#   書き方の例：CP4 = "〇〇だから，△△になると考えた．"
#   注意：CP4 に入れた文字列が 10 文字より短い（空のままも含む）と，エラーになって先へ進めない
CP4 = ""
checkpoint(CP4, "止まる 4")

## (12) 討議（5 分）

問いは授業ページ演習2にある．下の順位表を画面に出して，外れた予測から話す．

In [ ]:
print("=== 討議用: 4 指標の順位（1 = 最良）===")
ranks = pd.DataFrame({
    "accuracy":          table["accuracy"].rank(ascending=False).astype(int),
    "loss":              table["loss"].rank().astype(int),
    "infer_ms_per_1000": table["infer_ms_per_1000"].rank().astype(int),
    "latency_ms_b1":     table["latency_ms_b1"].rank().astype(int),
    "peak_mem_mb":       table["peak_mem_mb"].rank().astype(int),
})
display(ranks.rename(columns=JA).rename_axis("モデル"))
print("指標ごとに 1 位が違う数:", ranks.idxmin().nunique(), "モデル")
print("正解率の最大差:", f"{(table['accuracy'].max() - table['accuracy'].min())*100:.2f} ポイント（3 シードのばらつきは約 0.2〜0.7 ポイント）")